In [ ]:
#rodar no colab
#pip install deepwave==0.0.10

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 43 kB 189 kB/s 
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
    Preparing wheel metadata ... done
     |████████████████████████████████| 145 kB 25.0 MB/s 
  Created wheel for deepwave: filename=deepwave-0.0.10-py3-none-any.whl size=45121 sha256=5726979147efd4738b27d08a27663a20443ebf6200759c2a6260048af802bd22
  Stored in directory: /root/.cache/pip/wheels/de/44/ec/2fb562c6d43237c197f80cbb8afe7ec33335e5851704639cf7
Successfully built deepwave


In [1]:
import time
import torch
import numpy as np
import scipy.ndimage
import matplotlib.pyplot as plt
import deepwave
from deepwave import scalar
import cv2

In [ ]:
#para rodar no colab
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dx = 10.0 
dt = 0.0030 
nz = 128
ny = 128
nt = 640
peak_freq = 8
peak_time = 1.0 / peak_freq
num_shots = 12
num_sources_per_shot = 1
num_receivers_per_shot = 128 # um receptor a cada cela do modelo
source_spacing = 10 # em número de celas (100m)
receiver_spacing = 1 # em número de celas (10m)
first_source= 4 # borda para acomodar os 12 tiros
first_receiver=0
source_depth=1
receiver_depth=1

In [3]:
device = torch.device('cuda:0')

In [4]:
#geometria do levantamento
# source_locations
source_locations = torch.zeros(num_shots,num_sources_per_shot,2,
                               dtype=torch.long, device=device)
source_locations[..., 1] = source_depth
source_locations[:, 0, 0] = torch.arange(num_shots)*source_spacing+ first_source

# receiver_locations
receiver_locations = torch.zeros(num_shots,num_receivers_per_shot, 2,
                                 dtype=torch.long, device=device)
receiver_locations[..., 1] = receiver_depth
receiver_locations[:, :, 0] = ((torch.arange(num_receivers_per_shot)*receiver_spacing + first_receiver)
    .repeat(num_shots, 1))

# source_amplitudes
source_amplitudes = (deepwave.wavelets.ricker(peak_freq, nt, dt,peak_time)
    .repeat(num_shots,num_sources_per_shot, 1)
    .to(device))

In [5]:
#definições dos modelos
model_homo = torch.ones(ny, nz) * 1500.0 # modelo da água para excluir a onda direta
model_homogenous=model_homo.to(device)
# Propagate homogeneous
out = scalar(model_homogenous,dx,dt,source_amplitudes=source_amplitudes,
             source_locations=source_locations,
             receiver_locations=receiver_locations,
             accuracy=8,
             pml_freq=peak_freq,pml_width=[50,50,50,50])
receiver_amplitudes_homo=out[-1] #copia as amplitudes do dado de tiro

In [12]:
model_dir='../GeracaoModelosVel/Mods1/'
tiros_dir='./TirosMods1/'

In [14]:
for nummodel in range(1,501):
#carga modelo
  model=np.load(model_dir+"/Bin/model_{}.vp.npy".format(nummodel))
  model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
  model =torch.ones(ny, nz) * model_npy
  model_prop=model.to(device)
  # Propagate
  out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
              source_locations=source_locations,
              receiver_locations=receiver_locations,
              accuracy=8,
              pml_freq=peak_freq,pml_width=[50,50,50,50])
  receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
  np.save(tiros_dir+'/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
  print(nummodel)


C:\Users\Eder Dias\AppData\Local\Temp\ipykernel_25188\2386001057.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  model =torch.ones(ny, nz) * model_npy


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [ ]:
for nummodel in range(1,301):
#carga modelo
  model=np.load("/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos02/Bin/model_{}.vp.npy".format(nummodel))
  model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
  model =torch.ones(ny, nz) * model_npy
  model_prop=model.to(device)
  # Propagate
  out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
              source_locations=source_locations,
              receiver_locations=receiver_locations,
              accuracy=8,
              pml_freq=peak_freq,pml_width=[50,50,50,50])
  receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
  np.save('/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Shots02/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
  print(nummodel)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [ ]:
for nummodel in range(1,301):
#carga modelo
  model=np.load("/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos03/Bin/model_{}.vp.npy".format(nummodel))
  model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
  model =torch.ones(ny, nz) * model_npy
  model_prop=model.to(device)
  # Propagate
  out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
              source_locations=source_locations,
              receiver_locations=receiver_locations,
              accuracy=8,
              pml_freq=peak_freq,pml_width=[50,50,50,50])
  receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
  np.save('/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Shots03/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
  print(nummodel)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
